# This file is used to generate bowling statistics from raw match data.
 It calculates Overs, Economy, Average, and Bowling Impact.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [8]:
base_path = Path.cwd()
print("Current directory:",base_path)
data_path = base_path.parent / "data" / "02_cleaned_data"/ 'cleaned_data.csv'
print("Data path:",data_path)
dest_path = base_path.parent /'data'/ "02_cleaned_data"/ 'bowling_stats.csv'

Current directory: /home/mohan/Projects/wpl-2026-analysis/notebooks
Data path: /home/mohan/Projects/wpl-2026-analysis/data/02_cleaned_data/cleaned_data.csv


In [9]:
source_df = pd.read_csv(data_path)

In [ ]:
# create a new dataframe named bowling_stats
runs_df = source_df[
    ~(
        (source_df['runs_bat'] == 0) & (source_df['is_wide'] == 0) & (source_df['is_noball'] == 0)
    )
]

bowling_stats  = runs_df.groupby('bowler',as_index=False).agg({
    'runs_total':'sum'
})

In [ ]:
# Adding wickets column to bowling stats by removing runouts
wicket_df = source_df[
    source_df['wicket_kind'] != 'run out'
]

wicket_series  = wicket_df.groupby('bowler')['is_wicket'].sum()

bowling_stats['wickets'] = bowling_stats['bowler'].map(wicket_series)

In [ ]:
# Calculate Overs per bolwer
legal_deliveries_df = source_df[
    ( source_df['is_wide'] == 0 ) & (source_df['is_noball'] == 0)
]
legal_deliveries_series = legal_deliveries_df.groupby(['bowler'])['over'].count()

overs = (legal_deliveries_series //6) + (legal_deliveries_series % 6 ) /10

bowling_stats['overs'] = bowling_stats['bowler'].map(overs)

In [ ]:
# Calculate Economy of a bolwer
runs_series = bowling_stats.groupby('bowler')['runs_total'].sum()
overs = legal_deliveries_series/6
economy = np.round(np.divide(runs_series,overs),2)
bowling_stats['economy'] = bowling_stats['bowler'].map(economy)

In [72]:
# Calculate Dot Balls per bowler
dot_balls_series = source_df[
    (source_df['runs_bat'] == 0) & 
    (source_df['is_wide'] == 0) & 
    (source_df['is_noball'] == 0)
].groupby('bowler')['over'].count()


bowling_stats['dot_balls'] = bowling_stats['bowler'].map(dot_balls_series).fillna(0)

In [75]:
# Calculate Bowling Impact per bowler
bowling_stats['bowling_impact'] = (
    (bowling_stats['wickets'] * 20) + 
    (bowling_stats['dot_balls'] * 1) - 
    (bowling_stats['runs_total'] * 0.5)
)

In [77]:
bowling_stats.sort_values('bowling_impact',ascending=False)

,bowler,runs_total,wickets,overs,economy,dot_balls,bowling_impact
37,SFM Devine,272,17,32.5,8.28,79.0,283.0
16,LK Bell,199,12,36.0,5.53,128.0,268.5
21,N de Klerk,251,16,32.0,7.84,68.0,262.5
24,NSS Sharma,316,17,38.0,8.32,76.0,258.0
3,AC Kerr,210,14,28.0,7.50,65.0,240.0
4,CA Henry,249,14,29.0,8.59,73.0,228.5
18,M Kapp,256,10,40.0,6.40,122.0,194.0
27,RS Gayakwad,178,11,22.0,8.09,58.0,189.0
20,N Shree Charani,313,14,37.4,8.31,64.0,187.5
39,SR Patil,275,11,30.3,9.02,67.0,149.5


In [78]:
bowling_stats.to_csv(dest_path)